# 6.2b - Multihead CLS `pred_std3`

This notebook builds two text-only disagreement predictors from the fine-tuned CLS embeddings exported from the binary multihead notebook:

- `pred_std3_cv5`: 5-fold pooled cross-fit using the locked meeting folds from `6.2`
- `pred_std3_bert_test`: strict train-to-test prediction using the locked 70/15/15 BERT split

The pooled version gives out-of-sample Ridge predictions for all meetings. The strict BERT-test version keeps only the held-out meetings from the original multihead split.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from multihead_cls_pred_std3_utils import (
    build_bert_test_pred_std3,
    build_cv_pred_std3,
    ensure_bert_split_map,
    ensure_cv_fold_map,
    export_pred_frames,
    load_cls_features,
    load_turn_level_disagreement,
    summary_row,
)

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

ROOT = Path("/content") if IN_COLAB else Path("..").resolve()
OUTPUT_DIR = ROOT / "output" / "stance"

# Point these at the saved CLS export from the multihead notebook.
# The export should contain:
# - X_ft.npy
# - cls_meta.csv  (must include turn_uid)
CLS_DIR = OUTPUT_DIR
CLS_NPY_PATH = CLS_DIR / "X_ft.npy"
CLS_META_PATH = CLS_DIR / "cls_meta.csv"

RIDGE_ALPHA = 1.0

print(f"ROOT: {ROOT}")
print(f"CLS_NPY_PATH exists:  {CLS_NPY_PATH.exists()}")
print(f"CLS_META_PATH exists: {CLS_META_PATH.exists()}")
print(f"Locked fold CSV exists:  {(OUTPUT_DIR / 'disagreement_sgkf_folds.csv').exists()}")
print(f"Locked split CSV exists: {(OUTPUT_DIR / 'disagreement_bert_split.csv').exists()}")

In [ ]:
turns_wide = load_turn_level_disagreement(OUTPUT_DIR)
x_ft = load_cls_features(turns_wide, CLS_NPY_PATH, CLS_META_PATH)
fold_map = ensure_cv_fold_map(OUTPUT_DIR)
bert_split_map = ensure_bert_split_map(OUTPUT_DIR)

print(f"Turns: {len(turns_wide):,}")
print(f"Meetings: {turns_wide['doc_id'].nunique()}")
print(f"CLS shape: {x_ft.shape}")

In [ ]:
cv_turns = build_cv_pred_std3(turns_wide, x_ft, fold_map, alpha=RIDGE_ALPHA)
bert_turns = build_bert_test_pred_std3(turns_wide, x_ft, bert_split_map, alpha=RIDGE_ALPHA)

summary_df = pd.DataFrame([
    summary_row("cv5_pooled", cv_turns, pred_col="pred_std3_cv5", keep_all=True),
    summary_row("bert_test_only", bert_turns, pred_col="pred_std3_bert_test", keep_all=False),
])
summary_df

In [ ]:
cv_doc = (
    cv_turns.groupby(["bank", "doc_id"], as_index=False)
    .agg(
        date=("date", "first"),
        actual_score_std_3way=("score_std_3way", "mean"),
        pred_std3=("pred_std3_cv5", "mean"),
        n_turns=("turn_uid", "size"),
    )
    .sort_values(["date", "bank", "doc_id"])
)

bert_doc = (
    bert_turns[bert_turns["pred_std3_bert_test"].notna()]
    .groupby(["bank", "doc_id"], as_index=False)
    .agg(
        date=("date", "first"),
        actual_score_std_3way=("score_std_3way", "mean"),
        pred_std3=("pred_std3_bert_test", "mean"),
        n_turns=("turn_uid", "size"),
    )
    .sort_values(["date", "bank", "doc_id"])
)

print(f"CV5 pooled meetings: {len(cv_doc)}")
print(f"BERT test-only meetings: {len(bert_doc)}")
display(cv_doc.head())
display(bert_doc.head())

In [ ]:
saved_paths = export_pred_frames(OUTPUT_DIR, cv_turns=cv_turns, bert_turns=bert_turns)
pd.Series({k: str(v) for k, v in saved_paths.items()}, name="path")

## Handoff

Use these outputs in the LP regressions:

- `multihead_pred_std3_cv5_doc.csv`: full 206-meeting pooled 5-fold Option B
- `multihead_pred_std3_bert_test_doc.csv`: strict held-out-meeting version

Both files expose `doc_id` and `pred_std3` for direct merges with meeting-level shock data.